In [1]:
import numpy as np

In [2]:
class QuantizationNp:

  def __init__(self, num_bits=8):
    self.num_bits = num_bits
    # Determine target integer range
    if self.num_bits == 8:
        self.qmin, self.qmax = -128, 127
    elif self.num_bits == 16:
        self.qmin, self.qmax = -32768, 32767
    else:
        raise ValueError("num_bits must be 8 or 16")

    self.zero_point = self.scale = None

  def quantize(self, x: np.array) -> np.array:
    xmin, xmax = x.min(), x.max()
    self.scale = (xmax - xmin) / (self.qmax - self.qmin) if xmax != xmin else 1.0
    self.zero_point = np.round(self.qmin - xmin / self.scale)

    q = self.zero_point + x / self.scale
    q = np.round(q).astype(np.int32)
    q = np.clip(q, self.qmin, self.qmax).astype(np.int16 if self.num_bits == 16 else np.int8)

    return q

  def dequantize(self, q: np.array) -> np.array:
    if self.scale is None:
      raise ValueError("quantization parameters not set. You must call quantize() first.")
    return self.scale * (q.astype(np.float32) - self.zero_point)


In [3]:
x = np.array([0.1, -0.5, 1.2, -2.0], dtype=np.float32)
quantizer = QuantizationNp(num_bits=8)
q8 = quantizer.quantize(x)

print("quantized:", q8)
print("scale:", quantizer.scale)
print("zero_point:", quantizer.zero_point)

x32 = quantizer.dequantize(q8)
print("dequantized:", x32)

e = np.mean((x32 - x)**2)
print("MSE:", e)


quantized: [  39   -9  127 -128]
scale: 0.012549019
zero_point: 31.0
dequantized: [ 0.10039216 -0.50196075  1.2047058  -1.9952941 ]
MSE: 1.207208e-05


In [4]:
m = np.array([[2.52, -1.12, 1.74, 0.05], 
              [0.08, -0.22, -1.12, 2.65],
              [-0.13, 1.6, 0.22, -1.31],
              [2.13, -0.01, 1.83, 1.65]
             ])

quantizer = QuantizationNp(num_bits=8)
m8 = quantizer.quantize(m)

print("quantized:", m8)
print("scale:", quantizer.scale)
print("zero_point:", quantizer.zero_point)

m32 = quantizer.dequantize(m8)
print("dequantized:", m32)

e = np.mean((m32 - m)**2)
print("MSE:", e)

quantized: [[ 118 -116   68  -41]
 [ -39  -58 -116  127]
 [ -52   59  -30 -128]
 [  93  -45   74   62]]
scale: 0.015529411764705882
zero_point: -44.0
dequantized: [[ 2.51576471 -1.11811765  1.73929412  0.04658824]
 [ 0.07764706 -0.21741176 -1.11811765  2.65552941]
 [-0.12423529  1.59952941  0.21741176 -1.30447059]
 [ 2.12752941 -0.01552941  1.83247059  1.64611765]]
MSE: 1.3034602076124785e-05
